
#RNN FOR SENTIMENT CLASSIFICATION
---



##0.REFERENCE AND CONTEXT

**Notebook title: Recurrent Neural Networks for Sentiment Classification with IMDB Movie Reviews**

**Introduction**

Recurrent neural networks are one of the standard ways to teach machine learning models that process sequences. In earlier tutorials, we studied images and word embeddings. Images introduced spatial structure, which led naturally to convolutional neural networks. Text introduces another kind of structure: order. A movie review is not just a collection of words. It is a sequence of words, and the meaning of the review often depends on how those words unfold over time.

The IMDB movie review dataset is one of the classic open-source examples for teaching recurrent neural networks. The task is binary sentiment classification. Each review is labeled as positive or negative. The dataset is especially useful because it is already available through Keras, already tokenized into integer word identifiers, and small enough to run comfortably in Google Colab. This makes it ideal for an introductory notebook where the goal is not industrial-scale performance, but conceptual clarity.

The key teaching idea is that word order matters. Consider the sentence “the movie was good” and compare it with “the movie was not good.” Both sentences contain the word good, but they express different meanings. A model that simply averages word embeddings may not fully capture the role of negation or sequence. A recurrent neural network reads tokens one step at a time. As it reads, it updates an internal state. That state acts like a memory of what has been seen so far.

In this notebook, we use a Long Short-Term Memory network, or LSTM. An LSTM is a more powerful recurrent architecture designed to handle longer dependencies than a simple vanilla RNN. It uses internal gates to decide what information to keep, what information to forget, and what information to pass forward. This makes it a strong teaching example because it shows how neural networks can be designed to manage sequence memory.

The model begins with an embedding layer. Each word identifier is converted into a dense vector. The sequence of word vectors is then passed into an LSTM layer. The LSTM reads the review as an ordered sequence and produces a learned representation of the review. Dense layers then use this representation to predict whether the review is positive or negative. The final sigmoid output gives the probability of positive sentiment.

The notebook also includes a custom sentence testing section. This is important pedagogically because students can observe how the trained model responds to sentences such as “I loved this movie,” “I did not love this movie,” “This was not bad,” and “The acting was good but the ending was terrible.” These examples help make sequence learning concrete.

As in the rest of the project, the notebook follows the 10 cell plus 1 structure. The ten implementation cells handle environment setup, data loading, review inspection, preprocessing, dataset creation, model construction, training, evaluation, custom sentence testing, and artifact export. The additional LLM cell uses GPT-5.2 to explain the model and results from saved artifacts. The purpose is to teach not only model construction, but also disciplined, auditable interpretation.

##1.LIBRARIES AND ENVIRONMENT

**Explanation for Cell 1**

This cell prepares the notebook environment. It imports the required libraries, fixes random seeds, creates folders for artifacts and models, and records a run manifest. The manifest helps make the experiment reproducible and auditable by saving information about the environment and execution time.

In [1]:
import os
import re
import json
import random
import platform
import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

PROJECT_DIR = Path("/content/rnn_lstm_imdb_project")
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
MODEL_DIR = PROJECT_DIR / "model"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("run_%Y%m%d_%H%M%S_utc")

environment_manifest = {
    "run_id": RUN_ID,
    "created_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "tensorflow_version": tf.__version__,
    "numpy_version": np.__version__,
    "seed": SEED
}

manifest_path = ARTIFACT_DIR / "run_manifest.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(environment_manifest, f, indent=2)

print("Run ID:", RUN_ID)
print("TensorFlow version:", tf.__version__)
print("Artifacts folder:", ARTIFACT_DIR)

Run ID: run_20260604_180544_utc
TensorFlow version: 2.20.0
Artifacts folder: /content/rnn_lstm_imdb_project/artifacts


##2.THE IMBD DATASET

**Explanation for Cell 2**

This cell loads the IMDB movie review dataset. The reviews are already encoded as sequences of integer word identifiers. We limit the vocabulary to the most frequent words so the model remains small enough for teaching in Colab. The cell also builds dictionaries for translating between word identifiers and readable words.

In [2]:
VOCAB_SIZE = 10000
MAX_SEQUENCE_LENGTH = 250

(X_train_raw, y_train), (X_test_raw, y_test) = keras.datasets.imdb.load_data(
    num_words=VOCAB_SIZE
)

word_index = keras.datasets.imdb.get_word_index()

index_to_word = {
    index + 3: word
    for word, index in word_index.items()
}

index_to_word[0] = "<PAD>"
index_to_word[1] = "<START>"
index_to_word[2] = "<UNK>"
index_to_word[3] = "<UNUSED>"

word_to_index = {
    word: index
    for index, word in index_to_word.items()
}

dataset_summary = {
    "training_reviews": int(len(X_train_raw)),
    "test_reviews": int(len(X_test_raw)),
    "vocabulary_size_used": int(VOCAB_SIZE),
    "maximum_sequence_length": int(MAX_SEQUENCE_LENGTH),
    "label_meaning": {
        "0": "negative",
        "1": "positive"
    }
}

print(json.dumps(dataset_summary, indent=2))
print("First review token count:", len(X_train_raw[0]))
print("First review label:", int(y_train[0]))
print("First 30 integer tokens:")
print(X_train_raw[0][:30])

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
{
  "training_reviews": 25000,
  "test_reviews": 25000,
  "vocabulary_size_used": 10000,
  "maximum_sequence_length": 250,
  "label_meaning": {
    "0": "negative",
    "1": "positive"
  }
}
First review token count: 218
First review label: 1
First 30 integer tokens:
[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480]


##3.DATA INSPECTION

**Explanation for Cell 3**

This cell inspects the review data. It decodes one review into approximate text and calculates the distribution of review lengths. This matters because recurrent neural networks process sequences, and sequence length affects training speed, memory use, and the amount of context the model can read.

In [ ]:
def decode_review(token_sequence):
    words = [
        index_to_word.get(token_id, "<UNK>")
        for token_id in token_sequence
    ]
    return " ".join(words)

review_lengths = np.array([len(review) for review in X_train_raw])

length_summary = {
    "minimum_length": int(np.min(review_lengths)),
    "median_length": float(np.median(review_lengths)),
    "mean_length": float(np.mean(review_lengths)),
    "maximum_length": int(np.max(review_lengths)),
    "positive_training_reviews": int(np.sum(y_train == 1)),
    "negative_training_reviews": int(np.sum(y_train == 0))
}

print("Review length and label summary")
print(json.dumps(length_summary, indent=2))

print("\nDecoded example review")
print(decode_review(X_train_raw[0])[:1500])
print("\nLabel:", "positive" if y_train[0] == 1 else "negative")

plt.figure(figsize=(8, 5))
plt.hist(review_lengths, bins=50)
plt.xlabel("Review length in tokens")
plt.ylabel("Number of reviews")
plt.title("Distribution of IMDB Review Lengths")
plt.grid(True)

length_plot_path = ARTIFACT_DIR / "review_length_distribution.png"
plt.savefig(length_plot_path, dpi=150, bbox_inches="tight")
plt.show()

print("Saved review length distribution to:", length_plot_path)

##4.CELL PADDING AND DATA PREPARATION


**Explanation for Cell 4**

This cell pads and truncates the reviews. Neural networks train in batches, and batches require sequences with a common length. Short reviews are padded with zeros, while long reviews are truncated. We use post-padding and post-truncation so the beginning of each review is preserved in its original order.

In [3]:
X_train_padded = keras.preprocessing.sequence.pad_sequences(
    X_train_raw,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post",
    value=0
)

X_test_padded = keras.preprocessing.sequence.pad_sequences(
    X_test_raw,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post",
    value=0
)

y_train = y_train.astype("float32")
y_test = y_test.astype("float32")

X_train = X_train_padded[:20000]
X_val = X_train_padded[20000:]

y_train_main = y_train[:20000]
y_val = y_train[20000:]

split_summary = {
    "training_samples": int(X_train.shape[0]),
    "validation_samples": int(X_val.shape[0]),
    "test_samples": int(X_test_padded.shape[0]),
    "sequence_length": int(X_train.shape[1])
}

print(json.dumps(split_summary, indent=2))
print("Training matrix shape:", X_train.shape)
print("Validation matrix shape:", X_val.shape)
print("Test matrix shape:", X_test_padded.shape)

{
  "training_samples": 20000,
  "validation_samples": 5000,
  "test_samples": 25000,
  "sequence_length": 250
}
Training matrix shape: (20000, 250)
Validation matrix shape: (5000, 250)
Test matrix shape: (25000, 250)


##5.THE TENSORFLOW DATASETS

**Explanation for Cell 5**

This cell creates TensorFlow datasets. The training data is shuffled, batched, and prefetched. For sequence models, batching is especially useful because each batch contains many padded review sequences. The model processes these batches during training and updates its weights after each batch.

In [4]:
BATCH_SIZE = 64

train_ds = (
    tf.data.Dataset
    .from_tensor_slices((X_train, y_train_main))
    .shuffle(buffer_size=len(X_train), seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset
    .from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    tf.data.Dataset
    .from_tensor_slices((X_test_padded, y_test))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

for batch_X, batch_y in train_ds.take(1):
    print("Input batch shape:", batch_X.shape)
    print("Label batch shape:", batch_y.shape)
    print("First labels in batch:", batch_y[:10].numpy())

Input batch shape: (64, 250)
Label batch shape: (64,)
First labels in batch: [1. 0. 0. 0. 0. 1. 0. 1. 1. 1.]


##6.BUILDING THE RECURRENT NEURAL NETWORK

**Explanation for Cell 6**

This cell builds the recurrent neural network. The embedding layer converts word identifiers into dense vectors. The LSTM layer reads the sequence of word vectors one step at a time and builds a memory-based representation of the review. The dense layers then classify that representation as positive or negative sentiment.

In [5]:
EMBEDDING_DIM = 64
LSTM_UNITS = 64

def build_lstm_sentiment_model(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    lstm_units=LSTM_UNITS,
    sequence_length=MAX_SEQUENCE_LENGTH
):
    model = keras.Sequential(
        [
            layers.Input(shape=(sequence_length,), dtype="int32"),
            layers.Embedding(
                input_dim=vocab_size,
                output_dim=embedding_dim,
                mask_zero=True,
                name="word_embedding"
            ),
            layers.LSTM(
                lstm_units,
                dropout=0.20,
                recurrent_dropout=0.00,
                name="lstm_sequence_memory"
            ),
            layers.Dense(64, activation="relu", name="dense_sentiment_1"),
            layers.Dropout(0.30, name="dropout_1"),
            layers.Dense(1, activation="sigmoid", name="positive_probability")
        ],
        name="imdb_lstm_sentiment_classifier"
    )

    return model

model = build_lstm_sentiment_model()
model.summary()

Model: "imdb_lstm_sentiment_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ word_embedding (Embedding)      │ (None, 250, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_sequence_memory (LSTM)     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_sentiment_1 (Dense)       │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positive_probability (Dense)    │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 677,249 (2.58 MB)

 Trainable params: 677,249 (2.58 MB)

 Non-trainable params: 0 (0.00 B)

##7.COMPILATION AND TRAINING

**Explanation for Cell 7**

This cell compiles and trains the LSTM model. Binary cross-entropy is used because the task has two labels: negative and positive. The Adam optimizer updates the embedding vectors, LSTM weights, and dense classification layers. Early stopping restores the best validation model to reduce overfitting.

In [6]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=[early_stopping],
    verbose=1
)

history_dict = {
    key: [float(value) for value in values]
    for key, values in history.history.items()
}

history_path = ARTIFACT_DIR / "training_history.json"

with open(history_path, "w", encoding="utf-8") as f:
    json.dump(history_dict, f, indent=2)

print("Training history saved to:", history_path)

Epoch 1/12
313/313 ━━━━━━━━━━━━━━━━━━━━ 135s 408ms/step - accuracy: 0.7196 - loss: 0.5355 - val_accuracy: 0.8330 - val_loss: 0.4042
Epoch 2/12
313/313 ━━━━━━━━━━━━━━━━━━━━ 123s 392ms/step - accuracy: 0.8815 - loss: 0.3032 - val_accuracy: 0.8652 - val_loss: 0.3305
Epoch 3/12
  1/313 ━━━━━━━━━━━━━━━━━━━━ 1:22 265ms/step - accuracy: 0.8906 - loss: 0.2474

KeyboardInterrupt: 

##8.EVALUATION OF THE LSTM MODEL

**Explanation for Cell 8**

This cell evaluates the LSTM model. It plots the training and validation curves, computes test accuracy, prints a classification report, and displays a confusion matrix. These artifacts show whether the model learned a useful sequence-based sentiment classifier and whether it performs similarly on positive and negative reviews.

In [ ]:
epochs_ran = range(1, len(history.history["loss"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_ran, history.history["loss"], label="Training loss")
plt.plot(epochs_ran, history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)

loss_plot_path = ARTIFACT_DIR / "loss_curve.png"
plt.savefig(loss_plot_path, dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs_ran, history.history["accuracy"], label="Training accuracy")
plt.plot(epochs_ran, history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid(True)

accuracy_plot_path = ARTIFACT_DIR / "accuracy_curve.png"
plt.savefig(accuracy_plot_path, dpi=150, bbox_inches="tight")
plt.show()

test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)

y_prob = model.predict(X_test_padded, verbose=0).reshape(-1)
y_pred = (y_prob >= 0.50).astype("int32")

cm = confusion_matrix(y_test.astype("int32"), y_pred)

report_text = classification_report(
    y_test.astype("int32"),
    y_pred,
    target_names=["negative", "positive"]
)

report_dict = classification_report(
    y_test.astype("int32"),
    y_pred,
    target_names=["negative", "positive"],
    output_dict=True
)

evaluation_summary = {
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "test_accuracy_manual": float(accuracy_score(y_test.astype("int32"), y_pred)),
    "classification_report": report_dict,
    "confusion_matrix": cm.tolist()
}

evaluation_path = ARTIFACT_DIR / "evaluation_summary.json"

with open(evaluation_path, "w", encoding="utf-8") as f:
    json.dump(evaluation_summary, f, indent=2)

print("Test loss:", round(test_loss, 4))
print("Test accuracy:", round(test_accuracy, 4))
print("\nClassification report")
print(report_text)

plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix")
plt.colorbar()
plt.xticks([0, 1], ["negative", "positive"])
plt.yticks([0, 1], ["negative", "positive"])
plt.xlabel("Predicted label")
plt.ylabel("True label")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")

confusion_matrix_path = ARTIFACT_DIR / "confusion_matrix.png"
plt.savefig(confusion_matrix_path, dpi=150, bbox_inches="tight")
plt.show()

print("Saved evaluation summary to:", evaluation_path)

##9.TESTING

**Explanation for Cell 9**

This cell tests the trained model on short custom sentences. This is one of the clearest ways to teach why sequence models matter. Students can compare sentences with similar words but different order or negation, such as “this was good” and “this was not good.” The goal is to observe whether the LSTM responds differently to different sequential meanings.

In [ ]:
def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    tokens = text.split()
    return tokens

def encode_custom_sentence(text):
    tokens = simple_tokenize(text)

    encoded = [1]

    for token in tokens:
        token_id = word_to_index.get(token, 2)

        if token_id >= VOCAB_SIZE:
            token_id = 2

        encoded.append(token_id)

    padded = keras.preprocessing.sequence.pad_sequences(
        [encoded],
        maxlen=MAX_SEQUENCE_LENGTH,
        padding="post",
        truncating="post",
        value=0
    )

    return padded, tokens

custom_sentences = [
    "I loved this movie",
    "I did not love this movie",
    "This movie was good",
    "This movie was not good",
    "This movie was bad",
    "This movie was not bad",
    "The acting was good but the ending was terrible",
    "The acting was terrible but the ending was good",
    "A boring film with a wonderful final scene",
    "A wonderful film with a boring final scene"
]

custom_prediction_records = []

for sentence in custom_sentences:
    encoded_sentence, tokens = encode_custom_sentence(sentence)
    probability_positive = float(model.predict(encoded_sentence, verbose=0)[0][0])
    predicted_label = "positive" if probability_positive >= 0.50 else "negative"

    custom_prediction_records.append(
        {
            "sentence": sentence,
            "tokens": tokens,
            "positive_probability": probability_positive,
            "predicted_label": predicted_label
        }
    )

custom_predictions_path = ARTIFACT_DIR / "custom_sentence_predictions.json"

with open(custom_predictions_path, "w", encoding="utf-8") as f:
    json.dump(custom_prediction_records, f, indent=2)

for record in custom_prediction_records:
    print("Sentence:", record["sentence"])
    print("Positive probability:", round(record["positive_probability"], 4))
    print("Predicted label:", record["predicted_label"])
    print("-" * 60)

plt.figure(figsize=(9, 6))
sentence_labels = [
    "S" + str(i + 1)
    for i in range(len(custom_prediction_records))
]
probabilities = [
    record["positive_probability"]
    for record in custom_prediction_records
]

plt.bar(sentence_labels, probabilities)
plt.axhline(0.50, linestyle="--")
plt.xlabel("Custom sentence")
plt.ylabel("Positive sentiment probability")
plt.title("LSTM Predictions on Custom Sentences")
plt.grid(True)

custom_predictions_plot_path = ARTIFACT_DIR / "custom_sentence_predictions.png"
plt.savefig(custom_predictions_plot_path, dpi=150, bbox_inches="tight")
plt.show()

print("Saved custom sentence predictions to:", custom_predictions_path)
print("Saved custom sentence prediction plot to:", custom_predictions_plot_path)

##10.SAVING PREDICTIONS AND TRAINING PARAMETERS

**Explanation for Cell 10**

This cell saves prediction records and the trained model. It also stores a sample of incorrect predictions for later review. This completes the implementation workflow by preserving the model, numerical results, custom sentence tests, and error records as auditable artifacts.

In [ ]:
prediction_records = []

for i in range(len(y_test)):
    decoded_preview = decode_review(X_test_raw[i])[:500]

    record = {
        "test_index": int(i),
        "true_label_id": int(y_test[i]),
        "true_label": "positive" if int(y_test[i]) == 1 else "negative",
        "predicted_label_id": int(y_pred[i]),
        "predicted_label": "positive" if int(y_pred[i]) == 1 else "negative",
        "positive_probability": float(y_prob[i]),
        "correct": bool(y_pred[i] == int(y_test[i])),
        "decoded_review_preview": decoded_preview
    }

    prediction_records.append(record)

predictions_path = ARTIFACT_DIR / "prediction_records.json"

with open(predictions_path, "w", encoding="utf-8") as f:
    json.dump(prediction_records, f, indent=2)

incorrect_predictions = [
    record
    for record in prediction_records
    if record["correct"] is False
]

incorrect_predictions_path = ARTIFACT_DIR / "incorrect_prediction_examples.json"

with open(incorrect_predictions_path, "w", encoding="utf-8") as f:
    json.dump(incorrect_predictions[:50], f, indent=2)

model_path = MODEL_DIR / "imdb_lstm_sentiment_classifier.keras"
model.save(model_path)

print("Saved prediction records to:", predictions_path)
print("Saved incorrect prediction examples to:", incorrect_predictions_path)
print("Saved model to:", model_path)
print("Number of incorrect predictions:", len(incorrect_predictions))

##11.EXPLANATION AND SUMMARIZATION

**Explanation for the LLM Results Cell**

This cell uses GPT-5.2 to explain the LSTM model and its results. It reads the saved training history, evaluation summary, custom sentence predictions, and error examples. The output is requested as strict JSON so that facts, interpretation, assumptions, limitations, and recommended next steps are clearly separated. The API key is expected to be stored in Colab Secrets under OPENAI_API_KEY.

In [ ]:
import re
from google.colab import userdata
from openai import OpenAI

OPENAI_MODEL = "gpt-5.2"

api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY was not found in Colab Secrets.")

client = OpenAI(api_key=api_key)

with open(evaluation_path, "r", encoding="utf-8") as f:
    evaluation_for_llm = json.load(f)

with open(history_path, "r", encoding="utf-8") as f:
    history_for_llm = json.load(f)

with open(custom_predictions_path, "r", encoding="utf-8") as f:
    custom_predictions_for_llm = json.load(f)

with open(incorrect_predictions_path, "r", encoding="utf-8") as f:
    incorrect_predictions_for_llm = json.load(f)

compact_results = {
    "run_id": RUN_ID,
    "dataset_summary": dataset_summary,
    "split_summary": split_summary,
    "model_name": model.name,
    "embedding_dimension": EMBEDDING_DIM,
    "lstm_units": LSTM_UNITS,
    "epochs_completed": len(history_for_llm["loss"]),
    "final_training_loss": history_for_llm["loss"][-1],
    "final_validation_loss": history_for_llm["val_loss"][-1],
    "final_training_accuracy": history_for_llm["accuracy"][-1],
    "final_validation_accuracy": history_for_llm["val_accuracy"][-1],
    "test_loss": evaluation_for_llm["test_loss"],
    "test_accuracy": evaluation_for_llm["test_accuracy"],
    "classification_report": evaluation_for_llm["classification_report"],
    "confusion_matrix": evaluation_for_llm["confusion_matrix"],
    "custom_sentence_predictions": custom_predictions_for_llm,
    "sample_incorrect_predictions": incorrect_predictions_for_llm[:10]
}

system_prompt = """
You are a machine learning instructor.
Explain a recurrent neural network and LSTM sentiment classification experiment.
Return strict JSON only.
Do not invent results.
Separate observed facts from interpretation.
"""

user_prompt = f"""
Analyze the following experiment results.

Return JSON with this schema:

{{
  "plain_english_summary": "string",
  "why_imdb_is_good_for_teaching_rnn": "string",
  "rnn_concept_explanation": "string",
  "lstm_concept_explanation": "string",
  "model_explanation": "string",
  "results_explanation": "string",
  "custom_sentence_analysis": "string",
  "evidence_used": ["string"],
  "assumptions": ["string"],
  "limitations": ["string"],
  "recommended_next_steps": ["string"],
  "verification_status": "Not verified by external data"
}}

Experiment results:

BEGIN_RESULTS_JSON
{json.dumps(compact_results, indent=2)}
END_RESULTS_JSON
"""

response = client.responses.create(
    model=OPENAI_MODEL,
    input=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
)

llm_text = response.output_text.strip()

def extract_json_object(text):
    text = text.strip()
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    first = text.find("{")
    last = text.rfind("}")

    if first == -1 or last == -1 or last <= first:
        raise ValueError("No JSON object found in LLM response.")

    candidate = text[first:last + 1]
    return json.loads(candidate)

llm_explanation = extract_json_object(llm_text)

llm_explanation_path = ARTIFACT_DIR / "llm_results_explanation.json"

with open(llm_explanation_path, "w", encoding="utf-8") as f:
    json.dump(llm_explanation, f, indent=2)

print(json.dumps(llm_explanation, indent=2))
print("\nLLM explanation saved to:", llm_explanation_path)

##12.CONCLUSION

**Conclusion**

This notebook introduced recurrent neural networks through the IMDB movie review sentiment classification problem. It is a natural continuation of the earlier embedding tutorial because it uses the same broad type of data, but teaches a new and deeper idea. In the embedding notebook, the model learned vector representations of words and then summarized the review using average pooling. That approach is useful for teaching word embeddings, but it weakens the role of word order. The RNN tutorial adds sequence memory.

The central idea of a recurrent neural network is that data can arrive step by step. A review is not only a bag of words. It is an ordered sequence. The sentence “this movie was good” differs from “this movie was not good,” even though both contain many of the same words. Order, negation, contrast, and accumulation matter. A recurrent model is designed to read the sequence and update an internal state as each token arrives.

This notebook used an LSTM rather than a simple vanilla RNN. The LSTM is a standard teaching architecture because it addresses one of the weaknesses of ordinary recurrent networks: difficulty preserving useful information over longer sequences. The LSTM uses internal gates to regulate memory. It learns what to keep, what to forget, and what to pass forward. This gives students an intuitive model of sequence learning: the network reads, remembers, updates, and finally classifies.

The workflow followed the project’s 10 cell plus 1 structure. The notebook began with environment setup, data loading, review inspection, padding, and dataset construction. It then built the LSTM model using an embedding layer, an LSTM layer, dense classification layers, dropout, and a sigmoid output. The model was trained using binary cross-entropy and evaluated using accuracy, a classification report, and a confusion matrix.

The custom sentence testing cell is especially important for teaching. It allows students to examine how the model responds to short sentences designed around sentiment, negation, and contrast. Sentences such as “I loved this movie,” “I did not love this movie,” “This movie was not bad,” and “The acting was good but the ending was terrible” help move the lesson from abstract architecture to concrete linguistic behavior. The model may not always interpret these examples perfectly, and that itself is valuable. It shows both the power and limits of sequence models.

The notebook also preserves artifacts: training history, evaluation metrics, prediction records, custom sentence predictions, incorrect examples, and the trained model. This supports the governance-first teaching pattern used throughout the project. A machine learning notebook should not only produce a number; it should produce inspectable evidence.

The LLM explanation cell extends the workflow by asking GPT-5.2 to summarize the results in structured JSON. The language model does not replace evaluation. It explains the artifacts generated by the numerical model. This separation is important: the LSTM performs the classification, the saved metrics document performance, and the LLM provides a reviewable narrative interpretation.

This RNN tutorial therefore completes an important conceptual progression. Dense networks teach basic supervised learning. Embeddings teach learned representations of words. CNNs teach spatial structure in images. RNNs and LSTMs teach ordered sequence memory. Together, these notebooks give students a foundational map of deep learning architectures and the kinds of data structures each architecture was designed to handle.